### 🚀 Day 20 — ML Pipelines, ColumnTransformer & Data Leakage Prevention
Day 19 completed
Day 20 started
Week 3

Today we're moving from training individual ML models to building professional, reusable machine learning workflows.

Your main project context remains our Credit Card Fraud Detection dataset.
Today's learning objectives
By the end of Day 20, you'll understand:
1. Why preprocessing is necessary.
2. What an ML Pipeline is.
3. How ColumnTransformer works.
4. How to handle numerical and categorical columns.
5. What data leakage is and how to prevent it.
6. How to combine preprocessing and model training into one workflow.

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report 


In [2]:
# Step 2 — Create a sample dataset
data = {
    "Age":[22,25,47,52,46,56,23,40,35,60],
    "Salary": [25000,30000,60000,80000,55000,
               90000,28000,50000,45000,100000
        ],
        "Purchased": [0,0,1,1,1,1,0,1,0,1] 
}

df = pd.DataFrame(data)
print(df)

   Age  Salary  Purchased
0   22   25000          0
1   25   30000          0
2   47   60000          1
3   52   80000          1
4   46   55000          1
5   56   90000          1
6   23   28000          0
7   40   50000          1
8   35   45000          0
9   60  100000          1


In [3]:
# Our objective: predict whether a customer purchases a product.
# Step 3 — Separate X and y
X = df[["Age", "Salary"]]
y = df["Purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Step 4 — Create the Pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

# Understand the three steps:
# Step	Purpose
# imputer	Fills missing numerical values
# scaler	Standardizes numerical features
# model	Trains Logistic Regression

# Step 5 — Train and evaluate
pipeline.fit(X_train,y_train)
predictions = pipeline.predict(X_test)

print(classification_report(
    y_test,
    predictions,
    zero_division=0
))

# That's your first complete ML Pipeline!

# Important: This tiny dataset is only for learning the workflow. Its test metrics aren't reliable evidence of real-world performance.

              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3



In [5]:
# 3. Data Leakage — one of the most important ML concepts
# Before learning ColumnTransformer, understand this problem.

# Data leakage happens when information from outside the training data improperly influences model training.

# For example, suppose you have 10,000 records:
# - 8,000 training records
# - 2,000 test records
# You calculate the average salary using all 10,000 records, fill missing values, and then split the data.

#Split First
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

# Fit preprocessing only on the training data
scaler = StandardScaler()
X_trained_scaled = scaler.fit_transform(X_train)

# Only transform test data
X_test_scaled = scaler.transform(X_test)

# Remember these two methods:
# Method	Meaning
# fit()	Learn parameters from data
# transform()	Apply learned parameters to data
# fit_transform()	Learn and apply

# A Pipeline handles this correctly when you call fit() on training data. During prediction, it only applies the already-learned transformations.

In [ ]:
# 4. ColumnTransformer — handle different column types
# Real-world datasets often contain different types of features.

# Example:
# Age	Salary	City	Education
# 25	30000	Delhi	Graduate
# 35	60000	Mumbai	Postgraduate
# 42	80000	Pune	Graduate

# Problem:
# - Age and Salary are numerical.
# - City and Education are categorical.
# - Numerical features need numerical preprocessing.
# - Categorical features need encoding.
# This is where ColumnTransformer helps.

In [6]:
data = {
    "Age": [22, 25, 47, 52, 46, 56, 23, 40, 35, 60,
            28, 32, 45, 50, 29, 38, 42, 55, 31, 48],
    "Salary": [25000, 30000, 60000, 80000, 55000,
               90000, 28000, 50000, 45000, 100000,
               32000, 40000, 70000, 85000, 35000,
               52000, 65000, 95000, 42000, 75000],
    "City": ["Delhi", "Mumbai", "Pune", "Delhi", "Pune",
             "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi",
             "Pune", "Mumbai", "Delhi", "Pune", "Mumbai",
             "Delhi", "Pune", "Mumbai", "Delhi", "Pune"],
    "Education": ["Graduate", "Postgraduate", "Graduate",
                  "Postgraduate", "Graduate", "Postgraduate",
                  "Graduate", "Graduate", "Postgraduate",
                  "Postgraduate", "Graduate", "Graduate",
                  "Postgraduate", "Graduate", "Graduate",
                  "Postgraduate", "Graduate", "Postgraduate",
                  "Graduate", "Postgraduate"],
    "Purchased": [0, 0, 1, 1, 1, 1, 0, 1, 0, 1,
                  0, 0, 1, 1, 0, 1, 1, 1, 0, 1]
}

df = pd.DataFrame(data)

X = df.drop("Purchased", axis=1)
y = df["Purchased"]


# Step 2 — Identify numerical and categorical columns
numeric_features = ["Age", "Salary"]
categorical_features = ["City", "Education"]

In [7]:
# Step 3 — Create preprocessing pipelines
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# What's OneHotEncoder?

# It converts categories into numerical columns.

# For example:
# City	Delhi	Mumbai	Pune
# Delhi	1	0	0
# Mumbai	0	1	0
# Pune	0	0	1

# Step 4 — Combine using ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# This tells Scikit-learn:
# - Apply numerical preprocessing to Age and Salary.
# - Apply categorical preprocessing to City and Education.
# - Combine the resulting features.

# Step 5 — Create the complete ML Pipeline
full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Step 6 — Train and evaluate
X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

full_pipeline.fit(X_train,y_train)
predictions = full_pipeline.predict(X_test)
print(classification_report(
     y_test,predictions,zero_division=0
))

# You have now built a complete preprocessing and model-training workflow.

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



In [9]:
# 5. Apply today's learning to your Fraud Detection project
# Your Credit Card Fraud Detection dataset contains numerical features (V1–V28, Time, Amount).

# Therefore, you don't need categorical encoding for this dataset.

# Fit only on training data 

# You can create a clean pipeline like this:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

fraud_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Keep only numeric columns for your fraud pipeline
X_train_numeric = X_train.select_dtypes(include=['int64', 'float64'])
X_test_numeric = X_test.select_dtypes(include=['int64', 'float64'])

# Then fit your pipeline on the numeric data
fraud_pipeline.fit(X_train_numeric, y_train)
fraud_predictions = fraud_pipeline.predict(X_test_numeric)

print("Confusion Matrix:")
print(confusion_matrix(y_test, fraud_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    fraud_predictions,
    target_names=["Legitimate", "Fraud"],
    zero_division=0
))

Confusion Matrix:
[[2 0]
 [0 3]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00         2
       Fraud       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5

